# Machine Translation for Low-Resource Languages

**NLPAICS 2026 Summer School — The Paradigm Shift** · Day 2 · Tuesday 16 June 2026

**Lecturer:** Alicia Picazo Izquierdo

### 0 · Environment check

In [ ]:
# --- Environment check: run this cell first ---------------------------------
# It verifies you are on this lesson's kernel and that the GPU is visible.
import sys

assert ".venv" in sys.executable, (
    "Wrong kernel! In the menu choose: Kernel > Change Kernel > 'NLPAICS 05"
)
print("Kernel OK:", sys.executable)

try:
    import torch
    print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("torch not installed (fine if this lesson doesn't need it)")


Welcome to this practical session on Automatic Post-Editing (APE) for low-resource languages. In this session, you will build and evaluate a simple rule-based post-editor. Even if you have limited Python experience, don't worry! We'll guide you step-by-step.

### What is Automatic Post-Editing (APE)?

Machine Translation (MT) systems are constantly improving, but they don't always produce perfect translations, especially for languages with fewer available resources (low-resource languages). Human post-editors often correct these MT outputs. Automatic Post-Editing (APE) aims to automate this process by having a machine system automatically refine the MT output to bring it closer to a human-quality translation. This can save time and effort for human post-editors.

### Our Goal Today

You will implement a basic rule-based APE system. This system will learn simple rules from a dataset of Machine Translation outputs and their corresponding Human Post-Edits. It will then apply these learned rules to new MT outputs to improve them.

### The Dataset

We will be using a small sample from the [APE Shared Task 2022 synthetic dataset](https://github.com/surrey-nlp/APE-Shared-Task/tree/main/synthetic-data/ape-2022) for English-Marathi (en-mr). This dataset contains triplets of (Source Sentence, Machine Translation, Post-Edited Sentence).

Let's run our first cell!

In [ ]:
#!/usr/bin/env python3
"""
This is designed for a 30-minute practical session. It trains a tiny rule-based post-editor from (source, MT, post-edit) triplets and compares it with the
standard APE baseline: copying the raw MT output unchanged.

Main idea:
  1. Learn frequent whole-token substitutions from MT -> post-edit pairs.
  2. Learn frequent short phrase substitutions using SequenceMatcher.
  3. Apply only reliable substitutions to new MT sentences.
  4. Keep changes conservative: if no learned rule applies, copy the MT output.

It uses only the Python standard library.
"""

from __future__ import annotations

import argparse
import csv
import difflib
import math
import os
import random
import re
import shutil
import sys
import urllib.request
import zipfile
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

DATA_URL = "https://github.com/surrey-nlp/APE-Shared-Task/raw/main/synthetic-data/ape-2022/en-mr/en_mr_ape_synthetic_data.zip"
METHOD_NAME = "tiny_rule_ape"

SRC_KEYS = ("src", "source", "en")
MT_KEYS = ("mt", "target", "tgt", "hyp", "hypothesis", "machine")
PE_KEYS = ("pe", "post", "postedit", "post-edit", "ref", "reference")


def normalize_for_matching(text: str) -> str:
    """Light normalization used only for comparing strings."""
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text


def tokenize(text: str) -> List[str]:
    """Tokenize on whitespace; this is script-agnostic and works reasonably for Marathi."""
    return normalize_for_matching(text).split()


def detokenize(tokens: Sequence[str]) -> str:
    return " ".join(tokens)


def download_dataset(data_dir: Path, force: bool = False) -> Path:
    """Download and unzip the synthetic APE dataset if it is not already present."""
    data_dir.mkdir(parents=True, exist_ok=True)
    zip_path = data_dir / "en_mr_ape_synthetic_data.zip"
    extracted_marker = data_dir / ".download_complete"

    if extracted_marker.exists() and not force:
        print(f"Dataset already appears to be prepared in: {data_dir}")
        return data_dir

    if force and data_dir.exists():
        for item in data_dir.iterdir():
            if item.is_file() or item.is_symlink():
                item.unlink()
            else:
                shutil.rmtree(item)
        data_dir.mkdir(parents=True, exist_ok=True)

    print(f"Downloading data to {zip_path} ...")
    urllib.request.urlretrieve(DATA_URL, zip_path)

    print("Unzipping dataset ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(data_dir)
    extracted_marker.write_text("ok\n", encoding="utf-8")
    print(f"Data ready in: {data_dir}")
    return data_dir


def looks_like_tsv_triplets(path: Path) -> bool:
    try:
        with path.open("r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                if not line.strip():
                    continue
                return line.count("\t") >= 2
    except OSError:
        return False
    return False


def read_lines(path: Path) -> List[str]:
    with path.open("r", encoding="utf-8", errors="replace") as f:
        return [line.rstrip("\n") for line in f]


def score_name(path: Path, keys: Sequence[str]) -> int:
    name = path.name.lower()
    score = 0
    for key in keys:
        if re.search(rf"(^|[._\-]){re.escape(key)}($|[._\-])", name):
            score += 5
        elif key in name:
            score += 1
    return score


def find_parallel_files(root: Path) -> Tuple[Path, Path, Path] | None:
    """Find separate source, MT and PE files using flexible filename heuristics."""
    text_files = [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() not in {".zip", ".gz"}]
    candidates = []
    for p in text_files:
        try:
            if p.stat().st_size == 0 or p.stat().st_size > 2_000_000_000:
                continue
        except OSError:
            continue
        src_s = score_name(p, SRC_KEYS)
        mt_s = score_name(p, MT_KEYS)
        pe_s = score_name(p, PE_KEYS)
        if max(src_s, mt_s, pe_s) > 0:
            candidates.append((p, src_s, mt_s, pe_s))

    best = None
    best_score = -1
    for src, ss, _, _ in candidates:
        if ss <= 0:
            continue
        for mt, _, ms, _ in candidates:
            if mt == src or ms <= 0:
                continue
            for pe, _, _, ps in candidates:
                if pe in {src, mt} or ps <= 0:
                    continue
                # Prefer files in the same folder or with train in their names.
                folder_bonus = 5 if src.parent == mt.parent == pe.parent else 0
                train_bonus = sum("train" in p.name.lower() for p in (src, mt, pe))
                score = ss + ms + ps + folder_bonus + train_bonus
                if score > best_score:
                    best = (src, mt, pe)
                    best_score = score
    return best


def load_triplets_from_tsv(path: Path, max_examples: int | None = None) -> List[Tuple[str, str, str]]:
    rows = []
    with path.open("r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.reader(f, delimiter="\t")
        for parts in reader:
            if len(parts) < 3:
                continue
            rows.append((parts[0].strip(), parts[1].strip(), parts[2].strip()))
            if max_examples and len(rows) >= max_examples:
                break
    return rows


def load_triplets(data_dir: Path, max_examples: int | None = None) -> List[Tuple[str, str, str]]:
    """Load (source, MT, post-edit) triplets from separate files or TSV files."""
    separate = find_parallel_files(data_dir)
    if separate:
        src_file, mt_file, pe_file = separate
        print("Using separate files:")
        print(f"  source:    {src_file}")
        print(f"  MT:        {mt_file}")
        print(f"  post-edit: {pe_file}")
        src = read_lines(src_file)
        mt = read_lines(mt_file)
        pe = read_lines(pe_file)
        n = min(len(src), len(mt), len(pe))
        if max_examples:
            n = min(n, max_examples)
        return [(src[i], mt[i], pe[i]) for i in range(n)]

    tsv_files = [p for p in data_dir.rglob("*") if p.is_file() and looks_like_tsv_triplets(p)]
    if not tsv_files:
        raise FileNotFoundError(
            "Could not find APE triplets. Expected either .src/.mt/.pe-like files "
            "or a tab-separated file with source, MT and post-edit columns."
        )
    chosen = max(tsv_files, key=lambda p: p.stat().st_size)
    print(f"Using tab-separated triplet file: {chosen}")
    return load_triplets_from_tsv(chosen, max_examples=max_examples)


def split_data(triplets: List[Tuple[str, str, str]], dev_size: int, seed: int) -> Tuple[List, List]:
    rng = random.Random(seed)
    rows = triplets[:]
    rng.shuffle(rows)
    dev_size = max(1, min(dev_size, len(rows) // 5 if len(rows) >= 10 else len(rows) - 1))
    return rows[dev_size:], rows[:dev_size]


def collect_replacements(mt_tokens: List[str], pe_tokens: List[str], max_phrase_len: int) -> List[Tuple[str, str]]:
    """Extract candidate replacements from one MT/PE pair."""
    sm = difflib.SequenceMatcher(a=mt_tokens, b=pe_tokens, autojunk=False)
    replacements = []
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            continue
        old = mt_tokens[i1:i2]
        new = pe_tokens[j1:j2]
        if not old or not new:
            continue
        if len(old) <= max_phrase_len and len(new) <= max_phrase_len:
            replacements.append((detokenize(old), detokenize(new)))
    return replacements


def train_rule_post_editor(
    train: List[Tuple[str, str, str]],
    min_count: int = 2,
    min_precision: float = 0.65,
    max_phrase_len: int = 3,
) -> Dict[Tuple[str, ...], Tuple[str, ...]]:
    """Learn conservative phrase-level rewrite rules from MT to post-edits."""
    replacement_counts: Dict[str, Counter] = defaultdict(Counter)
    exposure_counts: Counter = Counter()

    for _src, mt, pe in train:
        mt_toks = tokenize(mt)
        pe_toks = tokenize(pe)
        for tok in set(mt_toks):
            exposure_counts[tok] += 1
        for old, new in collect_replacements(mt_toks, pe_toks, max_phrase_len=max_phrase_len):
            replacement_counts[old][new] += 1

    rules = {}
    for old, counter in replacement_counts.items():
        new, count = counter.most_common(1)[0]
        old_tokens = tuple(old.split())
        new_tokens = tuple(new.split())
        if not old_tokens or old_tokens == new_tokens:
            continue
        # For single words, estimate how often the old token should be changed.
        # For phrases, use replacement support only, because exact phrase exposure is rare.
        if len(old_tokens) == 1:
            exposure = max(exposure_counts[old_tokens[0]], 1)
            precision = count / exposure
            if count >= min_count and precision >= min_precision:
                rules[old_tokens] = new_tokens
        else:
            if count >= min_count:
                rules[old_tokens] = new_tokens

    print(f"Learned {len(rules)} rewrite rules.")
    return rules


def apply_rules(mt_sentence: str, rules: Dict[Tuple[str, ...], Tuple[str, ...]]) -> str:
    tokens = tokenize(mt_sentence)
    if not tokens:
        return mt_sentence.strip()
    max_len = max((len(k) for k in rules), default=1)
    out = []
    i = 0
    while i < len(tokens):
        changed = False
        for length in range(min(max_len, len(tokens) - i), 0, -1):
            key = tuple(tokens[i:i + length])
            if key in rules:
                out.extend(rules[key])
                i += length
                changed = True
                break
        if not changed:
            out.append(tokens[i])
            i += 1
    return detokenize(out)


def char_ngrams(text: str, n: int) -> Counter:
    text = normalize_for_matching(text)
    padded = " " * (n - 1) + text + " " * (n - 1)
    return Counter(padded[i:i + n] for i in range(max(0, len(padded) - n + 1)))


def corpus_chrf(preds: Sequence[str], refs: Sequence[str], max_n: int = 6, beta: float = 2.0) -> float:
    """A compact corpus chrF implementation for quick classroom feedback."""
    precisions, recalls = [], []
    for n in range(1, max_n + 1):
        pred_counts = Counter()
        ref_counts = Counter()
        for pred, ref in zip(preds, refs):
            pred_counts.update(char_ngrams(pred, n))
            ref_counts.update(char_ngrams(ref, n))
        overlap = sum((pred_counts & ref_counts).values())
        precision = overlap / max(sum(pred_counts.values()), 1)
        recall = overlap / max(sum(ref_counts.values()), 1)
        precisions.append(precision)
        recalls.append(recall)
    p = sum(precisions) / len(precisions)
    r = sum(recalls) / len(recalls)
    if p == 0 and r == 0:
        return 0.0
    beta2 = beta * beta
    return 100 * (1 + beta2) * p * r / (beta2 * p + r)


def exact_match_rate(preds: Sequence[str], refs: Sequence[str]) -> float:
    return 100 * sum(normalize_for_matching(p) == normalize_for_matching(r) for p, r in zip(preds, refs)) / max(len(refs), 1)


def write_submission(preds: Sequence[str], output_path: Path, method_name: str = METHOD_NAME) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as f:
        for i, pred in enumerate(preds, start=1):
            f.write(f"{method_name}\t{i}\t{pred}\n")


def show_examples(dev: Sequence[Tuple[str, str, str]], preds: Sequence[str], max_show: int = 5) -> None:
    print("\nExamples where the post-editor changed the MT output:")
    shown = 0
    for (src, mt, pe), pred in zip(dev, preds):
        if normalize_for_matching(mt) == normalize_for_matching(pred):
            continue
        shown += 1
        print("-" * 72)
        print(f"SRC: {src}")
        print(f"MT : {mt}")
        print(f"APE: {pred}")
        print(f"PE : {pe}")
        if shown >= max_show:
            break
    if shown == 0:
        print("No changed examples in this dev sample. Try lowering --min-count or increasing --train-size.")

def run_manual_demo(
    rules: Dict[Tuple[str, ...], Tuple[str, ...]],
    demo_cases: Sequence[Tuple[str, str]],
) -> None:
    """
    Apply the trained APE rules to a tiny manually written lecture sample.

    Each demo case contains:
        1. English source sentence
        2. manually supplied Marathi MT output

    Important: this script does not translate the English sentence. It only
    post-edits the MT output using rules learned from the real APE training data.
    Students do not need to provide Marathi references.
    """
    if not demo_cases:
        print("\nManual demo requested, but MANUAL_DEMO_CASES is empty.")
        return

    print("\nManual ambiguous-sentence demo")
    print("The APE system was trained normally on the downloaded triplets.")
    print("These examples are used only for testing and classroom discussion.")
    print("Students can edit MANUAL_DEMO_CASES to try their own ambiguous sentences.")
    print("This demo does not compute scores because no Marathi reference is required.")

    changed_count = 0
    for i, case in enumerate(demo_cases, start=1):
        if len(case) != 2:
            print("-" * 72)
            print(f"Example {i} was skipped because it should contain exactly 2 fields: (SRC, MT).")
            continue

        src, mt = case
        ape = apply_rules(mt, rules)
        changed = normalize_for_matching(mt) != normalize_for_matching(ape)
        changed_count += int(changed)

        print("-" * 72)
        print(f"Example {i}")
        print(f"SRC: {src}")
        print(f"MT : {mt}")
        print(f"APE: {ape}")

        if changed:
            print("Change: APE modified the MT output")
        else:
            print("Change: no learned rule applied")

    print("-" * 72)
    print(f"Manual demo summary: changed {changed_count} out of {len(demo_cases)} examples.")

def main(args_list: List[str] | None = None) -> None:
    parser = argparse.ArgumentParser(description="Train and run a tiny CPU-friendly APE post-editor.")
    parser.add_argument("--data-dir", type=Path, default=Path("data/ape_2022_en_mr"), help="Where the dataset is/will be stored.")
    parser.add_argument("--download", action="store_true", help="Download the synthetic dataset from GitHub before training.")
    parser.add_argument("--force-download", action="store_true", help="Delete existing data directory and download again.")
    parser.add_argument("--max-examples", type=int, default=1200, help="Maximum triplets to load from disk. Colab/classroom default keeps this fast on CPU.")
    parser.add_argument("--train-size", type=int, default=1000, help="Number of examples used for training. Colab/classroom default keeps this fast on CPU.")
    parser.add_argument("--dev-size", type=int, default=200, help="Number of examples used for quick evaluation.")
    parser.add_argument("--min-count", type=int, default=2, help="Minimum times a rewrite must be observed.")
    parser.add_argument("--min-precision", type=float, default=0.65, help="Minimum estimated precision for single-token rewrites.")
    parser.add_argument("--max-phrase-len", type=int, default=3, help="Maximum old/new phrase length learned as a rule.")
    parser.add_argument("--output", type=Path, default=Path("outputs/tiny_rule_ape.tsv"), help="Output file in WMT APE submission-style TSV format.")
    parser.add_argument("--seed", type=int, default=13, help="Random seed for the train/dev split.")
    parser.add_argument("--manual-demo", action="store_true", help="After normal training/evaluation, run the small manually written ambiguous-sentence demo.")

    # Preserve original sys.argv and replace for argparse parsing
    original_argv = sys.argv
    if args_list is None:
        sys.argv = [original_argv[0]]  # Only the script name
    else:
        sys.argv = [original_argv[0]] + args_list

    args = parser.parse_args()
    sys.argv = original_argv  # Restore original sys.argv

    if args.download or args.force_download:
        download_dataset(args.data_dir, force=args.force_download)

    triplets = load_triplets(args.data_dir, max_examples=args.max_examples)
    if len(triplets) < 2:
        raise ValueError("Need at least two triplets to train and evaluate.")

    if args.train_size + args.dev_size < len(triplets):
        triplets = triplets[: args.train_size + args.dev_size]
    train, dev = split_data(triplets, dev_size=args.dev_size, seed=args.seed)
    train = train[: args.train_size]

    print(f"Loaded {len(triplets)} triplets. Training on {len(train)} and evaluating on {len(dev)}.")

    rules = train_rule_post_editor(
        train,
        min_count=args.min_count,
        min_precision=args.min_precision,
        max_phrase_len=args.max_phrase_len,
    )

    dev_mt = [mt for _src, mt, _pe in dev]
    dev_refs = [pe for _src, _mt, pe in dev]
    ape_preds = [apply_rules(mt, rules) for _src, mt, _pe in dev]

    print("\nQuick evaluation on held-out sample")
    print("Higher chrF and exact-match are better.")
    print(f"Copy-MT baseline chrF: {corpus_chrf(dev_mt, dev_refs):6.2f}")
    print(f"Tiny APE system  chrF: {corpus_chrf(ape_preds, dev_refs):6.2f}")
    print(f"Copy-MT exact match:   {exact_match_rate(dev_mt, dev_refs):6.2f}%")
    print(f"Tiny APE exact match:  {exact_match_rate(ape_preds, dev_refs):6.2f}%")

    write_submission(ape_preds, args.output)
    print(f"\nWrote submission-style output to: {args.output}")
    show_examples(dev, ape_preds)

    if args.manual_demo:
        # MANUAL_DEMO_CASES is defined in the next notebook cell so students can edit it easily.
        # If that cell has not been run, use an empty list instead of raising an error.
        demo_cases = globals().get("MANUAL_DEMO_CASES", [])
        run_manual_demo(rules, demo_cases)


## ✍️ Manual ambiguous-sentence demo: English + MT only

The APE model is still trained on the normal downloaded English–Marathi APE data. This small sample is only used afterwards for classroom testing.

Students only need to edit two columns:

1. the English source sentence, preferably ambiguous;
2. a Marathi MT output, for example copied from an MT system.

The notebook does **not** translate English. It only post-edits the Marathi MT output in the second column using rules learned during training. Since students may not know Marathi, this version does not ask them to write a Marathi reference. It simply shows whether the trained APE system changes the MT output.


In [ ]:
# Students can edit this small demo set.
# Each tuple has two parts:
#   1. English source sentence, often ambiguous
#   2. manually supplied Marathi MT output, for example copied from an MT system
#
# The system will train normally first. Then it will apply the learned APE rules only to the MT column below.

MANUAL_DEMO_CASES = [
    (
        "I saw her duck.",
        "मी तिचे बदक पाहिले",
    ),
    (
        "The chicken is ready to eat.",
        "कोंबडी खाण्यास तयार आहे",
    ),
    (
        "Visiting relatives can be annoying.",
        "नातेवाईकांना भेटणे त्रासदायक असू शकते",
    ),
    (
        "She saw the man with the telescope.",
        "तिने दुर्बिणीसह माणूस पाहिला",
    ),
    (
        "They are flying planes.",
        "ते विमाने उडवत आहेत",
    ),
]


In [ ]:
# Running the post-editor with the specified arguments:
# --download: Ensures the dataset is downloaded.
# --train-size 1000: Uses 1000 examples for training the post-editor.
# --manual-demo: After normal evaluation, tests the trained APE rules on MANUAL_DEMO_CASES.
main(args_list=[
    '--download',
    '--train-size', '1000',
    '--manual-demo'
])


Unzipping dataset ...
Data ready in: data/ape_2022_en_mr
Using separate files:
  source:    data/ape_2022_en_mr/Entertainment/ie_entertainment_20210320_en_mr/en-mr/ie_entertainment_train.src
  MT:        data/ape_2022_en_mr/Entertainment/ie_entertainment_20210320_en_mr/en-mr/ie_entertainment_train.mt
  post-edit: data/ape_2022_en_mr/Entertainment/ie_entertainment_20210320_en_mr/en-mr/ie_entertainment_train.pe
Loaded 1200 triplets. Training on 1000 and evaluating on 200.
Learned 18 rewrite rules.

Quick evaluation on held-out sample
Higher chrF and exact-match are better.
Copy-MT baseline chrF:  68.83
Tiny APE system  chrF:  68.44
Copy-MT exact match:     1.00%
Tiny APE exact match:    0.00%

Wrote submission-style output to: outputs/tiny_rule_ape.tsv

Examples where the post-editor changed the MT output:
------------------------------------------------------------------------
SRC: Mallika Dua’s father Vinod Dua slams Akshay Kumar: I am going to screw this cretin
MT : मल्लिकाचे वडील विन